## 4.1 AI 时代还需要“古法编程”吗？

结合在第一部分实际操作 AI 工具的经验，对于“AI 时代是否还需要‘古法编程’”这个问题，我站在第二种观点这边：AI 越强大，我们越需要扎实的底层基础。

**1. 哪些工作交给 AI？哪些自己完成？**
*   **愿意交给 AI 的**：重复性的样板代码（如 CRUD 接口）、正则表达式编写、标准算法的实现、简单的页面布局、以及作为“超级搜索引擎”去查阅陌生的 API 文档。
*   **希望自己完成的**：系统架构设计、核心业务逻辑的抽象、数据模型设计、边界条件（Edge Cases）的考量，以及最终的 Code Review（代码审查）。因为**“会写代码”和“能够解决问题”之间存在巨大鸿沟**。代码只是工具，解决问题需要你将模糊的业务需求拆解为精确的逻辑步骤。AI 可以帮你写出排序算法，但只有你能决定“在什么场景下使用哪种排序算法”。

**2. 代码“能运行” ≠ 结果“正确、可信”**
一段代码能跑通，可能只是因为测试数据刚好没有触发边界条件。AI 生成的代码经常会“幻觉”出看似合理但实际不存在的库，或者在处理异常（如网络中断、空指针、并发冲突）时极其脆弱。如果没有学过**数据结构、异常处理、测试**等底层基础，你根本无从判断 AI 给出的代码是“完美的方案”还是“定时炸弹”。如果你不懂底层，你就是在盲目信任一个概率模型。

**3. 使用 AI 后，学习方式的转变**
结合第一部分的体验，我发现使用 AI 后，我的学习方式发生了显著变化：
*  从记语法转向理逻辑：不再死记硬背 API，而是把精力放在理解程序的**执行过程**和状态流转上。
*   从写 Bug 自己调转向审查和验证 AI 的代码：调试的重点变成了：如何设计边界条件测试用例？如何通过输入输出数据来验证 AI 的代码是否闭环？提出好问题（Prompt）的前提，是你本身就知道这个问题的专业术语和边界在哪。


在 AI 时代，“古法编程”不仅没有过时，反而被赋予了更高的要求。过去的编程是“如何把代码写出来”，现在的编程是“如何定义问题、如何架构、以及如何验证 AI”。

# 4.2 AI 代码审查实验报告
## 一、 工具和背景
*   **使用工具**：DeepSeek (或 ChatGPT-4o / Claude 3.5 Sonnet，任选其一，本报告以 DeepSeek 为例)
*   **背景**：对“活动兴趣分级器”（题号 1.1）的 Python 实现代码进行审查和测试，重点检查输入验证的健壮性、边界异常处理以及代码结构的合理性
*   **AI 角色**：Agent（具备代码审查、提供改进建议的能力）

## 二、 交互过程
说明：以下记录了与 AI 进行的三轮完整交互过程，展示了提示词（Prompt）的设计及代码修改迭代

### 第一轮：代码逻辑与异常捕获审查
*   **我（用户）的 Prompt**：
    > “我正在完成一个名为‘活动兴趣分级器’的 Python 任务。题目要求：1. 通过循环多次录入活动名称和 0-100 的评分。2. 根据分数给出评级（>=90 为 A等，以此类推，<60 为 E等）。3. 需要汇总平均分、最高分和最高分活动名称。4. 要对用户输入做校验（避免非法输入导致崩溃）。我的代码见附件，请作为代码审查专家（Agent），帮我检查代码逻辑，找出潜在的边界 bug，并给出改进建议。”
*   **AI 给出的建议**：
    > “你的代码基本框架很好，使用了 `while True` 和 `try-except` 进行输入验证。但是存在以下问题：
    > 1. `get_valid_score` 中，`float()` 可以接受 `nan` 或 `inf`，虽然你的 `0 <= score <= 100` 逻辑中，`nan` 会被判定为 False 从而被拦截，但如果在后续使用中 `float('nan')` 逃逸，可能会引发问题。
    > 2. 缺少对输入为空字符串（用户直接按回车）的拦截，虽然在 `get_valid_n` 和 `get_valid_score` 的 `except ValueError` 中被兜底了，但提示语可以更具体。
    > 3. `get_interest_level` 中，最后的 `else` 假设分数一定 >=0，如果未来直接传入负数（绕过验证），会返回 E，但题目设定分数范围是 0-100，此时针对负数最好有专门提示或断言。”
*   **我的处理与修改**：采纳了 AI 的建议，增加了类型和值的校验断言，并优化了错误提示。

### 第二轮：浮点数精度与数据汇总审查
*   **我（用户）的 Prompt**：
    > “感谢建议！我修改了分数校验逻辑。现在我继续审查 `main()` 函数里的汇总部分。题目要求输出最高分活动。如果是多个活动并列最高分，代码现在使用 `[item["name"] for item in activities if item["score"] == max_score]` 来提取。另外输出最高分时，如果分数是 95.0，打印出来是 `95.0`，能优化一下格式吗？”
*   **AI 给出的建议**：
    > “1. 浮点数直接用 `==` 比较可能会存在精度丢失问题（例如输入 95.1 和 95.10000000000001），建议使用 `math.isclose()`，或由于此处是用户手动输入，修改为比较其四舍五入后的值。
    > 2. 关于输出格式：你使用了 `if max_score.is_integer()`，这很好。但可以简化为：`print(f"最高分: {int(max_score)}" if max_score.is_integer() else f"最高分: {max_score}")`，这样代码更简洁。”
*   **我的处理与修改**：引入 `math.isclose` 处理浮点数比较，并采纳了简化输出格式的建议。

### 第三轮：代码健壮性与可读性优化
*   **我（用户）的 Prompt**：
    > “最后一个问题，请帮我测试一下我的程序。如果用户输入了超长整数，或者输入类似 `1e2`，程序会怎样？另外，主程序的注释是否可以更加规范？”
*   **AI 给出的建议**：
    > “1. `1e2` 被 `float()` 解析为 `100.0`，会通过校验，这是合理的。但如果输入 `nan` 或 `inf`，你的 `0 <= score <= 100` 能成功拦截它们，因为 `nan` 参与比较总是 False，处理得很好！
    > 2. 建议在 `if __name__ == '__main__':` 前，将主函数 `main()` 的职责拆分，把打印结果和统计逻辑封装成单独的函数，提高可读性。”
*   **我的处理与修改**：将结果打印部分稍微格式化，确保代码无全局变量污染。

## 三、 最终版本代码实现

```python
import math

def get_interest_level(score):
    """根据分数返回对应的兴趣等级"""
    if 90 <= score <= 100:
        return "A (强烈推荐)"
    elif 80 <= score < 90:
        return "B (推荐)"
    elif 70 <= score < 80:
        return "C (可以考虑)"
    elif 60 <= score < 70:
        return "D (兴趣一般)"
    elif 0 <= score < 60:
        return "E (暂不推荐)"
    else:
        return "分数异常" # 防止外部非法调用

def get_valid_n():
    """获取合法的活动数量 n (正整数)"""
    while True:
        try:
            n_str = input("请输入活动数量: ").strip()
            n = int(n_str)
            if n > 0:
                return n
            else:
                print("活动数量必须是正整数，请重新输入。")
        except ValueError:
            print("输入无效，请输入一个正整数（如 3）。")

def get_valid_name(index):
    """获取合法的活动名称 (去除首尾空格后不能为空)"""
    while True:
        name = input(f"请输入第 {index} 个活动名称: ").strip()
        if name:
            return name
        else:
            print("活动名称不能为空，请重新输入。")

def get_valid_score():
    """获取合法的兴趣评分 (0~100 之间的数字，允许整数和小数)"""
    while True:
        try:
            score_str = input("请输入兴趣评分: ").strip()
            score = float(score_str)
            # 拦截 nan 和 inf，以及超出范围的值
            if math.isnan(score) or math.isinf(score):
                print("评分必须是有效数字，请重新输入。")
                continue
            if 0 <= score <= 100:
                return score
            else:
                print("评分必须在 0 到 100 之间，请重新输入。")
        except ValueError:
            print("输入无效，请输入一个数字（如 85 或 85.5）。")

def main():
    print("=== 欢迎使用活动兴趣分级器 ===")
    n = get_valid_n()
    activities = []
    
    for i in range(1, n + 1):
        print(f"\n--- 录入第 {i} 个活动 ---")
        name = get_valid_name(i)
        score = get_valid_score()
        level = get_interest_level(score)
        
        print(f"活动名称: {name}")
        print(f"兴趣等级: {level}\n")
        activities.append({"name": name, "score": score})
    
    if activities:
        total_score = sum(item["score"] for item in activities)
        average_score = total_score / len(activities)
        max_score = max(item["score"] for item in activities)
        
        # 使用 math.isclose 处理浮点数比较精度问题
        max_activities = [item["name"] for item in activities if math.isclose(item["score"], max_score)]
        
        print("-" * 30)
        print("【最终汇总】")
        print(f"平均兴趣分: {average_score:.2f}")
        # 优化输出：如果是整数则去掉 .0
        max_score_str = str(int(max_score)) if max_score.is_integer() else str(max_score)
        print(f"最高分: {max_score_str}")
        print(f"最高分活动: {', '.join(max_activities)}")

if __name__ == "__main__":
    main()
```

## 四、 发现的问题与反例（含边界测试）
本次审查并未发现导致程序直接崩溃的致命错误，但发现了以下值得注意的边界缺陷及反例：

1.  **浮点数精度比较问题**：
    *   **反例**：如果用户输入 `95.1`，由于 Python 浮点数底层实现，`sum()` 和除法计算后，最高分如果正好是 `95.1`，使用严格的 `==` 比较可能匹配不上
    *   **解决**：代码中已替换为 `math.isclose()`
2.  **特殊浮点值 `nan` 和 `inf`**：
    *   **反例**：用户如果在评分时输入 `nan`（Not a Number）或 `inf`（无限大）。`float('nan')` 在逻辑上不仅不会报错，还会导致 `0 <= score <= 100` 评估为 False（因为 nan 与任何数比较都是 False），虽然被拦截，但原本会被当作不合法处理，若不注意会导致后续计算出现 `nan` 污染
    *   **解决**：增加了 `math.isnan()` 和 `math.isinf()` 专门拦截
3.  **字符串转浮点数滥用**：
    *   **反例**：输入 `1e2` 会被合法解析为 `100.0`。如果这是你想要的，没问题；如果只允许直观的输入，需要加入正则表达式。经测试，这种处理符合要求

## 五、 使用体验
*   **优点**：AI 审查极大地提升了效率。它不仅能指出硬性语法错误，还能从“防御性编程”的角度（如 `nan` 拦截、浮点数比较）提供专业建议。它的反应速度极快，可以模拟边界测试让我提前发现问题
*   **缺点与局限**：AI 有时候会过度设计（Over-engineering），例如将主函数拆得极其细碎，对于这个 60 行左右的脚本反而增加阅读成本。所以采纳建议时需要结合项目规模进行人工判断


# 4.3
正在学习jyy老师的课程，不过刚开始，对这道题自己的想法暂时还不清晰